# Unit 10 — Stacks, Queues & Deques

Is the bracket string `([{}])` balanced, and why does changing only its final bracket make `([{}]]` fail?
The answer depends on which opening bracket is still waiting most recently.
In this unit, a stack remembers the newest waiting item, a queue remembers the oldest waiting item, and Python's `deque` supports both patterns efficiently.

## Lesson 1 — LIFO Stacks and FIFO Queues

A **stack** follows last in, first out, or **LIFO**: the most recently pushed item is the first one popped.
Think of a stack of trays where you add and remove a tray at the same end.
A **queue** follows first in, first out, or **FIFO**: the earliest enqueued item is the first one dequeued.
Think of people joining the back of a line and leaving from the front.

## One Deque, Two Disciplined Patterns

Import the structure with `from collections import deque`, then create an empty one with `deque()`.
For a stack, use `stack.appendleft(value)` to **push** and `stack.popleft()` to **pop**.
The newest stack item is at the left end, so **peek** without removing it with `stack[0]`.
For a queue, use `queue.append(value)` to **enqueue** at the back and `queue.popleft()` to **dequeue** from the front.

In [ ]:
from collections import deque

def solve(data: str) -> str:
    tokens = data.split()
    stack = deque()
    answer_text = ""
    index = 0
    while index < len(tokens):
        command = tokens[index]
        if command == "PUSH":
            stack.appendleft(tokens[index + 1])
            index = index + 2
        else:
            if answer_text != "":
                answer_text = answer_text + "\n"
            if command == "PEEK":
                answer_text = answer_text + stack[0]
            else:
                answer_text = answer_text + stack.popleft()
            index = index + 1
    return answer_text

assert solve("PUSH red PUSH blue PEEK POP PEEK") == "blue\nblue\nred"

## A Queue Keeps Arrival Order

Enqueue at the right with `append`, but always dequeue the oldest item from the left with `popleft`.
If `Ava` arrives before `Bo`, a queue serves `Ava` first even though `Bo` was added more recently.
Using the stack push pattern here would reverse the service order and solve a different problem.

In [ ]:
from collections import deque

def solve(data: str) -> str:
    tokens = data.split()
    queue = deque()
    served_text = ""
    index = 0
    while index < len(tokens):
        command = tokens[index]
        if command == "ENQUEUE":
            queue.append(tokens[index + 1])
            index = index + 2
        else:
            if served_text != "":
                served_text = served_text + "\n"
            served_text = served_text + queue.popleft()
            index = index + 1
    return served_text

assert solve("ENQUEUE Ava ENQUEUE Bo DEQUEUE ENQUEUE Cy DEQUEUE") == "Ava\nBo"

## Lesson 2 — Match Brackets with a Stack

Scan a bracket string from left to right and push every opening bracket.
For a closing bracket, the stack must be nonempty and `stack[0]` must be its matching opening bracket; then pop that opening bracket.
A closing bracket with no opener fails immediately, but leftover openers are discovered only after the entire string has been scanned.
The empty string is balanced because it has no unmatched bracket.

In [ ]:
from collections import deque

def solve(data: str) -> str:
    expression = data.strip()
    stack = deque()
    for character in expression:
        if character in "([{":
            stack.appendleft(character)
        else:
            if len(stack) == 0:
                return "NO"
            expected = ""
            if character == ")":
                expected = "("
            elif character == "]":
                expected = "["
            else:
                expected = "{"
            if stack[0] != expected:
                return "NO"
            stack.popleft()
    if len(stack) == 0:
        return "YES"
    return "NO"

assert solve("([{}])") == "YES"
assert solve("([{}]]") == "NO"
assert solve("") == "YES"

## Lesson 3 — Evaluate Postfix Expressions

In postfix notation, also called **RPN**, an operator comes after its two operands: `7 2 -` means `7 - 2`.
Push each integer as it appears.
When an operator appears, pop the right operand first, pop the left operand second, compute `left operator right`, and push the result.
After the final token, exactly one value should remain; subtraction makes the operand order especially important.

In [ ]:
from collections import deque

def solve(data: str) -> str:
    tokens = data.split()
    stack = deque()
    for token in tokens:
        if token in "+-*" and len(token) == 1:
            right = stack.popleft()
            left = stack.popleft()
            if token == "+":
                value = left + right
            elif token == "-":
                value = left - right
            else:
                value = left * right
            stack.appendleft(value)
        else:
            stack.appendleft(int(token))
    return str(stack[0])

assert solve("8 3 - 2 * 5 +") == "15"
assert solve("7 2 -") == "5"

## A Stack-and-Queue Checklist

First, decide whether the next item to leave must be the newest item or the oldest item.
Use stack `appendleft` and `popleft` for LIFO work, and use queue `append` and `popleft` for FIFO work.
Before popping or peeking, handle the possibility that the deque is empty.
For monotonic stacks, compare against `stack[0]`; for RPN, remember that the first popped value is the right operand.

## Submit the Solver

After `solve(data)` works, a contest submission can use this wrapper.
It is marked `no-exec` because notebook execution has no contest input waiting for it.

In [ ]:
import sys
print(solve(sys.stdin.read()))